# Azure AI Agent Service Basics - Getting Started

Welcome to the Azure AI Agent Service tutorial! This notebook will guide you through the fundamentals of building AI agents using Microsoft's cloud-hosted agent platform.

## What is Azure AI Agent Service?

Azure AI Agent Service is a fully managed platform for building and deploying AI agents. Unlike self-hosted solutions, it provides:

- **Persistent State**: Agent threads and conversations are stored securely in Azure
- **Built-in Tools**: File Search, Code Interpreter, Bing Grounding, and more
- **Enterprise Security**: Role-based access control, content safety, and compliance
- **Scalability**: Automatic scaling and high availability

### Key Concepts:

- **Agent**: An AI assistant with specific instructions, tools, and capabilities
- **Thread**: A conversation session between a user and an agent (persisted in Azure)
- **Message**: A single turn in a conversation (user or assistant)
- **Run**: An execution of the agent to generate a response
- **Tool**: A capability the agent can use (functions, file search, code interpreter)

## Resources
- [Official Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/)
- [Python SDK Reference](https://learn.microsoft.com/en-us/python/api/overview/azure/ai-agents-readme)
- [Azure AI Foundry Portal](https://ai.azure.com)

Let's get started!

## Installation and Setup

First, let's install the required packages.

In [ ]:
# Install required packages
# !pip install azure-ai-projects azure-identity python-dotenv

## Environment Configuration

You'll need to configure your Azure AI Foundry project. Create a `.env` file with:

```env
# Azure AI Foundry Project
PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com

# Model deployment name
MODEL_DEPLOYMENT_NAME=gpt-4o
```

### Getting Your Project Endpoint

1. Go to [Azure AI Foundry](https://ai.azure.com)
2. Navigate to your project
3. Go to Project Settings > Overview
4. Copy the "Project connection string" or endpoint URL

In [ ]:
import os
from dotenv import load_dotenv

# Azure AI Projects SDK
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    MessageRole,
    FunctionTool,
    ToolSet,
    RunStatus
)
from azure.identity import DefaultAzureCredential

# Load environment variables
load_dotenv()

print("✅ Imports loaded successfully!")

In [ ]:
# Initialize the Azure AI Project Client
# This client provides access to agents, threads, and other AI services

project_client = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint=os.environ["PROJECT_ENDPOINT"]
)

# Get the model deployment name
MODEL_DEPLOYMENT = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o")

print(f"✅ Connected to Azure AI Foundry project")
print(f"📦 Using model: {MODEL_DEPLOYMENT}")

## 1. Creating Your First Agent

An agent in Azure AI Agent Service is an AI assistant with:
- **Name**: A human-readable identifier
- **Instructions**: System prompt that defines behavior
- **Model**: The language model to use
- **Tools**: Optional capabilities (function calling, file search, etc.)

In [ ]:
# Create a simple agent
agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="TutorialAssistant",
    instructions="""You are a helpful assistant that provides clear, concise answers.
    Always be friendly and informative. If you don't know something, say so.
    Keep responses focused and to the point."""
)

print(f"✅ Agent created!")
print(f"   ID: {agent.id}")
print(f"   Name: {agent.name}")
print(f"   Model: {agent.model}")

## 2. Working with Threads

A **Thread** represents a conversation session. Unlike local chat histories, threads in Azure AI Agent Service are:
- **Persistent**: Stored securely in Azure
- **Resumable**: Can be continued later with full context
- **Shareable**: Can be accessed by different processes/users with proper permissions

In [ ]:
# Create a new thread
thread = project_client.agents.create_thread()

print(f"✅ Thread created!")
print(f"   ID: {thread.id}")
print(f"   Created: {thread.created_at}")

## 3. Sending Messages and Running the Agent

The conversation flow is:
1. Create a message in the thread
2. Create a run to process the message with the agent
3. Retrieve the response

In [ ]:
# Add a user message to the thread
message = project_client.agents.create_message(
    thread_id=thread.id,
    role=MessageRole.USER,
    content="Hello! What can you help me with today?"
)

print(f"✅ Message added to thread")
print(f"   Role: {message.role}")
print(f"   Content: {message.content[0].text.value}")

In [ ]:
# Run the agent on the thread
# This processes the message and generates a response
run = project_client.agents.create_and_process_run(
    thread_id=thread.id,
    agent_id=agent.id
)

print(f"✅ Run completed!")
print(f"   Run ID: {run.id}")
print(f"   Status: {run.status}")

In [ ]:
# Retrieve the assistant's response
messages = project_client.agents.list_messages(thread_id=thread.id)

# Get the most recent assistant message
last_message = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)

if last_message:
    print("🤖 Assistant Response:")
    print(last_message.text.value)
else:
    print("⚠️ No response from assistant")

## 4. Continuing the Conversation

Threads maintain conversation history, so subsequent messages have full context.

In [ ]:
# Helper function for conversation
def chat(user_message: str, thread_id: str, agent_id: str) -> str:
    """Send a message and get a response."""
    # Add user message
    project_client.agents.create_message(
        thread_id=thread_id,
        role=MessageRole.USER,
        content=user_message
    )
    
    # Run the agent
    run = project_client.agents.create_and_process_run(
        thread_id=thread_id,
        agent_id=agent_id
    )
    
    # Get response
    messages = project_client.agents.list_messages(thread_id=thread_id)
    response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
    
    return response.text.value if response else "No response"

print("✅ Chat helper function created!")

In [ ]:
# Continue the conversation
response = chat("Can you tell me about the weather in Paris?", thread.id, agent.id)
print("🤖 Assistant:", response)
print("\n" + "-"*50 + "\n")

response = chat("What about things to do there?", thread.id, agent.id)
print("🤖 Assistant:", response)
print("\n" + "-"*50 + "\n")

# Test context retention
response = chat("Which city are we talking about?", thread.id, agent.id)
print("🤖 Assistant:", response)

## 5. Viewing Thread History

You can retrieve the full conversation history from a thread.

In [ ]:
# List all messages in the thread
messages = project_client.agents.list_messages(thread_id=thread.id)

print("📜 Full Conversation History:")
print("=" * 60)

# Messages are in reverse chronological order, so we reverse them
for msg in reversed(list(messages)):
    role = "👤 User" if msg.role == MessageRole.USER else "🤖 Assistant"
    # Get the text content
    content = msg.content[0].text.value if msg.content else "[No content]"
    # Truncate long messages for display
    if len(content) > 200:
        content = content[:200] + "..."
    print(f"{role}: {content}")
    print("-" * 40)

## 6. Function Calling with Custom Tools

One of the most powerful features is the ability to give agents custom tools through function calling. The agent can decide when to call your functions based on the conversation.

In [ ]:
import json

# Define custom functions that the agent can call
def get_weather(city: str) -> str:
    """Get current weather for a city (simulated)."""
    weather_data = {
        "paris": {"temp": 18, "condition": "Partly cloudy", "humidity": 65},
        "london": {"temp": 14, "condition": "Rainy", "humidity": 80},
        "tokyo": {"temp": 22, "condition": "Sunny", "humidity": 55},
        "new york": {"temp": 20, "condition": "Clear", "humidity": 50},
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        data = weather_data[city_lower]
        return f"Weather in {city}: {data['temp']}°C, {data['condition']}, Humidity: {data['humidity']}%"
    return f"Weather data not available for {city}"

def get_time(timezone: str) -> str:
    """Get current time in a timezone (simulated)."""
    from datetime import datetime, timedelta
    offsets = {
        "utc": 0, "est": -5, "pst": -8, "cet": 1, "jst": 9
    }
    tz = timezone.lower()
    if tz in offsets:
        utc_time = datetime.utcnow()
        local_time = utc_time + timedelta(hours=offsets[tz])
        return f"Current time in {timezone.upper()}: {local_time.strftime('%Y-%m-%d %H:%M:%S')}"
    return f"Unknown timezone: {timezone}"

# Map function names to implementations
FUNCTIONS = {
    "get_weather": get_weather,
    "get_time": get_time
}

print("✅ Custom functions defined!")

In [ ]:
# Define the function schemas for the agent
weather_function = FunctionTool(
    name="get_weather",
    description="Get the current weather for a specified city",
    parameters={
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city to get weather for"
            }
        },
        "required": ["city"]
    }
)

time_function = FunctionTool(
    name="get_time",
    description="Get the current time in a specified timezone",
    parameters={
        "type": "object",
        "properties": {
            "timezone": {
                "type": "string",
                "description": "The timezone abbreviation (e.g., UTC, EST, PST, CET, JST)"
            }
        },
        "required": ["timezone"]
    }
)

print("✅ Function schemas defined!")

In [ ]:
# Create a new agent with tools
toolset = ToolSet()
toolset.add(weather_function)
toolset.add(time_function)

tool_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="ToolAssistant",
    instructions="""You are a helpful assistant with access to tools.
    Use the get_weather tool when users ask about weather in a city.
    Use the get_time tool when users ask about the current time.
    Always use the tools when relevant - don't make up information.""",
    toolset=toolset
)

print(f"✅ Agent with tools created!")
print(f"   ID: {tool_agent.id}")
print(f"   Tools: {[t.function.name for t in tool_agent.tools if hasattr(t, 'function')]}")

In [ ]:
from azure.ai.projects.models import (
    RequiredFunctionToolCall,
    ToolOutput,
    SubmitToolOutputsAction
)

def run_with_tools(user_message: str, thread_id: str, agent_id: str) -> str:
    """Run agent with tool calling support."""
    # Add user message
    project_client.agents.create_message(
        thread_id=thread_id,
        role=MessageRole.USER,
        content=user_message
    )
    
    # Create run
    run = project_client.agents.create_run(
        thread_id=thread_id,
        agent_id=agent_id
    )
    
    # Process the run with tool calls
    while run.status in [RunStatus.QUEUED, RunStatus.IN_PROGRESS, RunStatus.REQUIRES_ACTION]:
        run = project_client.agents.get_run(thread_id=thread_id, run_id=run.id)
        
        if run.status == RunStatus.REQUIRES_ACTION:
            if isinstance(run.required_action, SubmitToolOutputsAction):
                tool_outputs = []
                
                for tool_call in run.required_action.submit_tool_outputs.tool_calls:
                    if isinstance(tool_call, RequiredFunctionToolCall):
                        func_name = tool_call.function.name
                        args = json.loads(tool_call.function.arguments)
                        
                        print(f"  📞 Calling: {func_name}({args})")
                        
                        # Execute the function
                        if func_name in FUNCTIONS:
                            result = FUNCTIONS[func_name](**args)
                            print(f"  📋 Result: {result}")
                        else:
                            result = f"Unknown function: {func_name}"
                        
                        tool_outputs.append(
                            ToolOutput(tool_call_id=tool_call.id, output=result)
                        )
                
                # Submit tool outputs
                run = project_client.agents.submit_tool_outputs_to_run(
                    thread_id=thread_id,
                    run_id=run.id,
                    tool_outputs=tool_outputs
                )
    
    # Get response
    messages = project_client.agents.list_messages(thread_id=thread_id)
    response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
    
    return response.text.value if response else "No response"

print("✅ Tool-aware run function created!")

In [ ]:
# Create a new thread for the tool agent
tool_thread = project_client.agents.create_thread()

# Test weather function
print("👤 User: What's the weather like in Tokyo?")
response = run_with_tools("What's the weather like in Tokyo?", tool_thread.id, tool_agent.id)
print(f"🤖 Assistant: {response}")
print("\n" + "="*60 + "\n")

# Test time function
print("👤 User: What time is it in EST?")
response = run_with_tools("What time is it in EST?", tool_thread.id, tool_agent.id)
print(f"🤖 Assistant: {response}")
print("\n" + "="*60 + "\n")

# Test combined usage
print("👤 User: I'm planning a trip to Paris. What's the weather there and what time is it in CET?")
response = run_with_tools(
    "I'm planning a trip to Paris. What's the weather there and what time is it in CET?", 
    tool_thread.id, 
    tool_agent.id
)
print(f"🤖 Assistant: {response}")

## 7. Agent with Structured Instructions

Let's create a more specialized agent with detailed instructions, similar to the restaurant host example from Semantic Kernel.

In [ ]:
# Define restaurant-specific functions
def get_menu_specials() -> str:
    """Get today's special menu items."""
    return json.dumps({
        "soup": {"name": "Clam Chowder", "price": "$12.99"},
        "salad": {"name": "Caesar Salad with Grilled Chicken", "price": "$14.99"},
        "drink": {"name": "Fresh Mint Lemonade", "price": "$4.99"},
        "dessert": {"name": "Chocolate Lava Cake", "price": "$8.99"}
    })

def check_reservation(name: str, date: str) -> str:
    """Check if a reservation exists."""
    # Simulated reservations
    reservations = {
        "smith": {"date": "2024-12-25", "time": "19:00", "party_size": 4},
        "johnson": {"date": "2024-12-24", "time": "18:30", "party_size": 2}
    }
    name_lower = name.lower()
    if name_lower in reservations:
        r = reservations[name_lower]
        return f"Found reservation for {name}: {r['date']} at {r['time']} for {r['party_size']} guests"
    return f"No reservation found for {name}"

def make_reservation(name: str, date: str, time: str, party_size: int) -> str:
    """Make a new reservation."""
    return f"✅ Reservation confirmed for {name}: {date} at {time} for {party_size} guests. Confirmation #R{hash(name) % 10000:04d}"

# Update function map
FUNCTIONS.update({
    "get_menu_specials": get_menu_specials,
    "check_reservation": check_reservation,
    "make_reservation": make_reservation
})

print("✅ Restaurant functions defined!")

In [ ]:
# Create function tool schemas
menu_function = FunctionTool(
    name="get_menu_specials",
    description="Get today's special menu items including soup, salad, drink, and dessert",
    parameters={"type": "object", "properties": {}}
)

check_reservation_function = FunctionTool(
    name="check_reservation",
    description="Check if a reservation exists for a given name and date",
    parameters={
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "Guest's last name"},
            "date": {"type": "string", "description": "Date in YYYY-MM-DD format"}
        },
        "required": ["name", "date"]
    }
)

make_reservation_function = FunctionTool(
    name="make_reservation",
    description="Make a new restaurant reservation",
    parameters={
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "Guest's name for the reservation"},
            "date": {"type": "string", "description": "Date in YYYY-MM-DD format"},
            "time": {"type": "string", "description": "Time in HH:MM format"},
            "party_size": {"type": "integer", "description": "Number of guests"}
        },
        "required": ["name", "date", "time", "party_size"]
    }
)

# Create toolset
restaurant_toolset = ToolSet()
restaurant_toolset.add(menu_function)
restaurant_toolset.add(check_reservation_function)
restaurant_toolset.add(make_reservation_function)

print("✅ Restaurant tool schemas created!")

In [ ]:
# Create the restaurant host agent
restaurant_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="RestaurantHost",
    instructions="""You are a friendly and professional restaurant host at "The Azure Bistro".
    
Your responsibilities:
1. Welcome guests warmly
2. Help with menu inquiries using the get_menu_specials tool
3. Check existing reservations using check_reservation tool
4. Make new reservations using make_reservation tool

Guidelines:
- Always be polite and welcoming
- Use tools when guests ask about menu, reservations
- Confirm details before making reservations
- If a guest asks about specials, always show the full list
- Thank guests for choosing The Azure Bistro""",
    toolset=restaurant_toolset
)

print(f"✅ Restaurant Host agent created!")
print(f"   ID: {restaurant_agent.id}")

In [ ]:
# Test the restaurant agent
restaurant_thread = project_client.agents.create_thread()

conversations = [
    "Hi! I'd like to know about today's specials.",
    "I have a reservation under Smith. Can you check it?",
    "I'd like to make a reservation for 6 people on December 31st at 7:30 PM under the name Anderson."
]

for user_input in conversations:
    print(f"👤 Customer: {user_input}")
    response = run_with_tools(user_input, restaurant_thread.id, restaurant_agent.id)
    print(f"🍽️ Host: {response}")
    print("\n" + "="*60 + "\n")

## 8. Cleanup

It's good practice to clean up agents and threads when you're done, especially in development.

In [ ]:
# Clean up threads
threads_to_delete = [thread.id, tool_thread.id, restaurant_thread.id]

for thread_id in threads_to_delete:
    try:
        project_client.agents.delete_thread(thread_id)
        print(f"✅ Deleted thread: {thread_id}")
    except Exception as e:
        print(f"⚠️ Could not delete thread {thread_id}: {e}")

# Clean up agents
agents_to_delete = [agent.id, tool_agent.id, restaurant_agent.id]

for agent_id in agents_to_delete:
    try:
        project_client.agents.delete_agent(agent_id)
        print(f"✅ Deleted agent: {agent_id}")
    except Exception as e:
        print(f"⚠️ Could not delete agent {agent_id}: {e}")

## 🎉 Congratulations!

You've completed the Azure AI Agent Service Basics tutorial! Here's what you've learned:

### ✅ Key Concepts Covered:
1. **Project Client** - Connecting to Azure AI Foundry
2. **Agents** - Creating AI assistants with custom instructions
3. **Threads** - Managing persistent conversation sessions
4. **Messages & Runs** - The conversation execution flow
5. **Function Calling** - Giving agents custom tools and capabilities
6. **Tool Execution** - Handling tool calls and submitting results

### 🔧 Key Differences from Semantic Kernel:
| Semantic Kernel | Azure AI Agent Service |
|----------------|------------------------|
| Local chat history | Persistent threads in Azure |
| Plugin classes | Function tools with JSON schemas |
| `get_response()` | `create_and_process_run()` |
| Manual state management | Built-in state persistence |

### 🚀 Next Steps:
Continue with the **Agent Tools & Capabilities** notebook to learn about:
- File Search with vector stores
- Code Interpreter for running Python
- Bing Grounding for web search
- Streaming responses

### 📚 Additional Resources:
- [Azure AI Agent Service Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/)
- [Python SDK Reference](https://learn.microsoft.com/en-us/python/api/overview/azure/ai-agents-readme)
- [Azure AI Foundry Samples](https://github.com/azure-ai-foundry/foundry-samples)

Happy building! 🚀